## Fake News Classifier Using LSTM

Dataset: https://www.kaggle.com/c/fake-news/data#

In [16]:
import pandas as pd

In [17]:
df =pd.read_csv('fake_train.csv', sep=';')

In [18]:
df.head()

,Unnamed: 0,title,text,label
0,0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s...",1
1,1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...,1
2,2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...,0
3,3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...,1
4,4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...,0


In [19]:
df.shape

(24353, 4)

In [20]:
df.isnull().sum()

,0
Unnamed: 0,0
title,0
text,0
label,0


In [21]:
###Drop Nan Values
df=df.dropna()
## since this is a text data, we cannot replace the missing values with anything so we will drop


In [22]:
## Get the Independent Features

X=df.drop('label',axis=1)

In [23]:
## Get the Dependent features
y=df['label']

In [24]:
X.shape

(24353, 3)

In [25]:
y.shape

(24353,)

In [26]:
import tensorflow as tf

In [27]:
tf.__version__

'2.20.0'

In [28]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [29]:
### Vocabulary size
voc_size=5000

### Onehot Representation

In [30]:
messages=X.copy()

In [31]:
messages['title'][1]

"China says Trump call with Taiwan president won't change island's status"

In [32]:
messages

,Unnamed: 0,title,text
0,0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s..."
1,1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...
2,2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...
3,3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...
4,4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...
...,...,...,...
24348,24348,Mexico Senate committee OK's air transport dea...,MEXICO CITY (Reuters) - A key committee in Mex...
24349,24349,BREAKING: HILLARY CLINTON’S STATE DEPARTMENT G...,IF SHE S NOT TOAST NOW THEN WE RE IN BIGGER TR...
24350,24350,trump breaks from stump speech to admire beaut...,kremlin nato was created for agression \nruss...
24351,24351,NFL PLAYER Delivers Courageous Message: Stop B...,Dallas Cowboys star wide receiver Dez Bryant t...


In [33]:
#messages.reset_index(inplace=True)

In [34]:
messages

,Unnamed: 0,title,text
0,0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s..."
1,1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...
2,2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...
3,3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...
4,4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...
...,...,...,...
24348,24348,Mexico Senate committee OK's air transport dea...,MEXICO CITY (Reuters) - A key committee in Mex...
24349,24349,BREAKING: HILLARY CLINTON’S STATE DEPARTMENT G...,IF SHE S NOT TOAST NOW THEN WE RE IN BIGGER TR...
24350,24350,trump breaks from stump speech to admire beaut...,kremlin nato was created for agression \nruss...
24351,24351,NFL PLAYER Delivers Courageous Message: Stop B...,Dallas Cowboys star wide receiver Dez Bryant t...


In [35]:
import nltk
import re
from nltk.corpus import stopwords

In [36]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [37]:
## Data preprocessing

In [38]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

corpus=[]
for i in range(0,len(messages)):
  review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
  review = review.lower()
  review= review.split()
  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)


#This is a list comprehension. It iterates through every word, removes stopwords using a condition, and applies Porter stemming to the remaining words.

In [39]:
corpus

['palestinian switch christma light bethlehem anti trump protest',
 'china say trump call taiwan presid chang island statu',
 'fail trump organ credit score make laugh',
 'zimbabw militari chief china trip normal visit beij say',
 'uncourag presid ever receiv courag award proce whine current presid',
 'suspect boko haram suicid bomber kill least nigeria offici',
 'watch john oliv present gop debat clowntown f ck world sh tshow',
 'senat democrat ask trump attorney gener pick recus russia probe',
 'trump humili republican latest hissi fit side democrat debt ceil',
 'maci get boot loyal custom fire trump',
 'north korea nuclear test pacif logic terrifi',
 'czech polic ask parliament allow prosecut prospect pm babi',
 'one democrat refus cast elector vote crook hillari could end video',
 'watch senat al franken rip ted cruz new one insult senat hear',
 'nearli half american still oppos republican tax bill reuter ipso poll',
 'white hous declin say trump made decis climat accord',
 'u appe

In [40]:
corpus[1]

'china say trump call taiwan presid chang island statu'

In [41]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[1856, 1146, 1633, 2574, 3894, 1287, 1391, 725],
 [891, 1632, 1391, 4274, 1692, 4081, 2631, 4187, 2235],
 [2503, 1391, 1877, 3312, 2708, 4922, 3255],
 [1889, 2577, 3259, 891, 2319, 2740, 972, 2422, 1632],
 [1533, 4081, 992, 2518, 3254, 1334, 3548, 2997, 1338, 4081],
 [314, 4197, 4863, 1098, 4967, 2090, 902, 1115, 1363],
 [4642, 4834, 1785, 878, 4114, 184, 3575, 3137, 2215, 4427, 2409, 1308],
 [3983, 2154, 977, 1391, 1162, 1144, 2899, 4579, 4300, 91],
 [1391, 2495, 3887, 1577, 3297, 4303, 3010, 2154, 1721, 3576],
 [283, 4897, 2, 4302, 1415, 4631, 1391],
 [3807, 3472, 2074, 2196, 793, 271, 2733],
 [2755, 4162, 977, 1000, 87, 1656, 37, 3018, 2621],
 [1326, 2154, 3640, 1661, 202, 385, 2343, 4844, 2036, 1777, 2920],
 [4642, 3983, 787, 4862, 1929, 3488, 4856, 945, 1326, 1240, 3983, 401],
 [2602, 4466, 1379, 345, 4957, 3887, 3790, 835, 1440, 3333, 527],
 [3601, 3632, 3381, 1632, 1391, 4397, 2420, 2794, 4315],
 [3742, 2051, 824, 401, 3555, 3027, 3954, 1940, 1097],
 [4741, 695, 3183, 2154, 192

In [42]:
corpus[1]

'china say trump call taiwan presid chang island statu'

In [43]:
onehot_repr[1]

[891, 1632, 1391, 4274, 1692, 4081, 2631, 4187, 2235]

### Embedding Representation

In [44]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='post',maxlen=sent_length)
print(embedded_docs)

[[1856 1146 1633 ...    0    0    0]
 [ 891 1632 1391 ...    0    0    0]
 [2503 1391 1877 ...    0    0    0]
 ...
 [1391 3341 1133 ...    0    0    0]
 [2539 1499 2495 ...    0    0    0]
 [ 851 3661  777 ...    0    0    0]]


In [45]:
embedded_docs[1]

array([ 891, 1632, 1391, 4274, 1692, 4081, 2631, 4187, 2235,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [46]:
embedded_docs[0]

array([1856, 1146, 1633, 2574, 3894, 1287, 1391,  725,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [62]:
## Creating model
embedding_vector_features=40 ##features representation
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_shape=(sent_length,)))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,501 (1001.96 KB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

None


In [63]:
len(embedded_docs),y.shape

(24353, (24353,))

In [64]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [65]:
X_final.shape,y_final.shape

((24353, 20), (24353,))

In [66]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

### Model Training

In [92]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=20,batch_size=64)

Epoch 1/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9720 - loss: 0.0718 - val_accuracy: 0.9045 - val_loss: 0.4035
Epoch 2/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9769 - loss: 0.0611 - val_accuracy: 0.9031 - val_loss: 0.4174
Epoch 3/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9773 - loss: 0.0581 - val_accuracy: 0.8984 - val_loss: 0.3770
Epoch 4/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9807 - loss: 0.0503 - val_accuracy: 0.9006 - val_loss: 0.4041
Epoch 5/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9828 - loss: 0.0465 - val_accuracy: 0.9013 - val_loss: 0.4198
Epoch 6/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9836 - loss: 0.0437 - val_accuracy: 0.9008 - val_loss: 0.4832
Epoch 7/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9848 - loss: 0.0393 - val_accuracy: 0.8986 - val_loss: 0.4674
Epoch 8/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9873 - loss: 0.0341 - val_accuracy: 0

### Adding Dropout

In [ ]:
# from tensorflow.keras.layers import Dropout
# ## Creating model
# embedding_vector_features=40
# model=Sequential()
# model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
# model.add(Dropout(0.3))
# model.add(LSTM(100))
# model.add(Dropout(0.3))
# model.add(Dense(1,activation='sigmoid'))
# model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

### Performance Metrics And Accuracy

In [93]:
y_pred=model.predict(X_test)

153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [94]:
y_pred=np.where(y_pred > 0.5, 1,0) ##AUC ROC Curve

In [95]:
from sklearn.metrics import confusion_matrix

In [96]:
confusion_matrix(y_test,y_pred)

array([[1966,  249],
       [ 231, 2425]])

In [97]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9014576062410182

In [98]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.89      0.89      0.89      2215
           1       0.91      0.91      0.91      2656

    accuracy                           0.90      4871
   macro avg       0.90      0.90      0.90      4871
weighted avg       0.90      0.90      0.90      4871



In [60]:
0